# tt-mlir #8570 repro check (v2 — colab-9295 검증완료 노트북 구조 재사용)

런타임: **CPU 고RAM** 선택 → **런타임 → 모두 실행**. 완료되면 **파일 → GitHub에 사본 저장**.

In [ ]:
!nproc
!free -h

In [ ]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq clang ninja-build cmake git python3.12-venv

In [ ]:
%cd /content
!rm -rf /content/tt-mlir
!git clone --branch colab-8570-repro https://github.com/alexxony/tt-mlir.git /content/tt-mlir
%cd /content/tt-mlir
!git log --oneline -3

In [ ]:
import os
os.environ["TTMLIR_TOOLCHAIN_DIR"] = "/opt/ttmlir-toolchain/"
!mkdir -p /opt/ttmlir-toolchain
!chown -R $(whoami) /opt/ttmlir-toolchain

In [ ]:
%cd /content/tt-mlir
!cmake -B env/build env -DCMAKE_C_COMPILER=clang -DCMAKE_CXX_COMPILER=clang++
!cmake --build env/build

In [ ]:
%cd /content/tt-mlir
!bash -c "source env/activate && cmake -G Ninja -B build -DCMAKE_BUILD_TYPE=Release -DTTMLIR_ENABLE_STABLEHLO=ON -DTTMLIR_ENABLE_RUNTIME=OFF -DTTMLIR_ENABLE_RUNTIME_TESTS=OFF -DTTMLIR_ENABLE_OPMODEL=OFF -DTTMLIR_ENABLE_BINDINGS_PYTHON=OFF -DCMAKE_BUILD_PARALLEL_LEVEL=$(nproc)"
!bash -c "source env/activate && cmake --build build --target ttmlir-opt -- -j$(nproc)" 2>&1 | tee /content/main_build.log | tail -150

In [ ]:
import subprocess, os
r = subprocess.run(["tail", "-n", "60", "/content/main_build.log"], capture_output=True, text=True)
print(r.stdout)
print("ttmlir-opt exists:", os.path.exists("/content/tt-mlir/build/bin/ttmlir-opt"))

In [ ]:
repro_mlir = r"""// SPDX-FileCopyrightText: (c) 2026 Tenstorrent AI ULC
//
// SPDX-License-Identifier: Apache-2.0

// REQUIRES: stablehlo
// RUN: ttmlir-opt --convert-stablehlo-to-ttir %s | FileCheck %s

// Repro for https://github.com/tenstorrent/tt-mlir/issues/8570
// scatter with leading update_window_dims=[0] (Qwen 3.5 27B MRoPE position_ids pattern)

module @SyncTensorsGraph.45 attributes {mhlo.is_dynamic = false} {
  func.func @main(%arg0: tensor<3x1x494xi64>,
                  %arg1: tensor<3x494xi64>,
                  %arg2: tensor<494xi64>) -> tensor<3x1x494xi64> {
    %c   = stablehlo.constant dense<494> : tensor<494xi64>
    %c_0 = stablehlo.constant dense<0>   : tensor<494xi64>
    %0 = stablehlo.reshape %arg0 : (tensor<3x1x494xi64>) -> tensor<3x494xi64>
    %1 = stablehlo.reshape %arg2 : (tensor<494xi64>)     -> tensor<1x1x494xi64>
    %2 = stablehlo.reshape %1    : (tensor<1x1x494xi64>) -> tensor<494xi64>
    %3 = stablehlo.compare LT, %2, %c_0 : (tensor<494xi64>, tensor<494xi64>) -> tensor<494xi1>
    %4 = stablehlo.add %2, %c    : tensor<494xi64>
    %5 = stablehlo.select %3, %4, %2 : tensor<494xi1>, tensor<494xi64>
    %6 = stablehlo.reshape %5    : (tensor<494xi64>)     -> tensor<494x1xi64>
    %7 = stablehlo.reshape %arg1 : (tensor<3x494xi64>)   -> tensor<1x3x494xi64>
    %8 = stablehlo.reshape %7    : (tensor<1x3x494xi64>) -> tensor<3x494xi64>
    %9 = "stablehlo.scatter"(%0, %6, %8) <{
           scatter_dimension_numbers = #stablehlo.scatter<
             update_window_dims = [0], inserted_window_dims = [1],
             scatter_dims_to_operand_dims = [1], index_vector_dim = 1>
         }> ({
      ^bb0(%a: tensor<i64>, %b: tensor<i64>):
        stablehlo.return %b : tensor<i64>
      }) : (tensor<3x494xi64>, tensor<494x1xi64>, tensor<3x494xi64>) -> tensor<3x494xi64>
    %10 = stablehlo.reshape %9 : (tensor<3x494xi64>) -> tensor<3x1x494xi64>
    return %10 : tensor<3x1x494xi64>
  }
}
"""
with open("/content/repro_8570.mlir", "w") as f:
    f.write(repro_mlir)
print("written")

In [ ]:
%cd /content/tt-mlir
import subprocess
r = subprocess.run(
    ["bash", "-c", "source env/activate && ttmlir-opt --convert-stablehlo-to-ttir /content/repro_8570.mlir"],
    capture_output=True, text=True, timeout=60,
)
print("=== rc:", r.returncode, "===")
print("=== stdout ===")
print(r.stdout)
print("=== stderr ===")
print(r.stderr)

has_ttir_scatter = "ttir.scatter" in r.stdout
has_stablehlo_scatter = "stablehlo.scatter" in r.stdout
print()
print("=== 판정 ===")
print("ttir.scatter 존재:", has_ttir_scatter)
print("stablehlo.scatter 잔존:", has_stablehlo_scatter)
if r.returncode == 0 and has_ttir_scatter and not has_stablehlo_scatter:
    print(">>> 정상 lowering 완료 -- #8570 이미 고쳐짐")
elif r.returncode != 0:
    print(">>> rc!=0 -- 에러/verifier abort, #8570 아직 재현됨. stderr 확인")
else:
    print(">>> rc==0인데 패턴 불명확 -- 수동 확인 필요")